# Train π0.5 (pi05) on RunPod — FR5 pick-and-place

Fine-tunes the **π0.5** policy on the FR5 LeRobot dataset using this repo's shared trainer
(`common/train.py --policy pi05`). π0.5: same architecture as π0 but tokenizer_max_length=200 (richer language context).

## Pod setup (before opening this notebook)
| | |
|---|---|
| **Template** | RunPod **PyTorch 2.x** (CUDA ≥ 12.1) |
| **GPU** | 48 GB (A40 / A6000 / L40S) recommended · 24 GB (4090) works with `train_expert_only` · 80 GB (A100/H100) = full finetune, larger batch |
| **Disk** | ≥ 60 GB volume ( ~5 GB PaliGemma + dataset + checkpoints ) |

## One-time prerequisites
1. **HF token** (read + write) with the **PaliGemma license accepted**: <https://huggingface.co/google/paligemma-3b-pt-224> — the weights are gated; training cannot start without this.
2. **Dataset on the Hub**: push once from wherever the 150-episode dataset lives:
   `python tools/push_dataset_hf.py --root lerobot_dataset --repo <you>/fr5-pick-place-lerobot`
3. If the GitHub repo is private, a GitHub token with repo-read scope.

## VRAM cheat-sheet (bf16 + gradient checkpointing are on by default)
| GPU | memory_mode | batch_size |
|---|---|---|
| 24 GB | `expert_only` (freeze VLM, train 300M expert) | 2–4 |
| 48 GB | `freeze_vision` or `full` | 2 (full) / 4 (freeze_vision) |
| 80 GB | `full` | 4–8 |

> `expert_only` is not just a memory fallback — with only 150 episodes it's also the
> least-overfitting choice, mirroring how Octo is finetuned head-only here.

In [ ]:
# ── 1. Parameters — edit these ────────────────────────────────────────────────
import os

HF_TOKEN        = os.environ.get("HF_TOKEN", "")        # hf_... (read+write, PaliGemma licence accepted)
HF_DATASET_REPO = "<you>/fr5-pick-place-lerobot"        # pushed with tools/push_dataset_hf.py
GIT_URL         = "https://github.com/SreevaatsavB/fairino-fr5-policies.git"
GIT_TOKEN       = os.environ.get("GIT_TOKEN", "")       # only if the repo is private
GIT_BRANCH      = "main"

POLICY       = "pi05"
MEMORY_MODE  = "auto"      # auto | full | freeze_vision | expert_only
BATCH_SIZE   = None        # None -> picked from GPU VRAM (see cheat-sheet)
MAX_EPOCHS   = 100
PROPRIO_MODE = "full"      # full | dropout | none  (benchmark axis)

WORKSPACE = "/workspace"
REPO_DIR  = f"{WORKSPACE}/fairino-fr5-policies"
DATA_DIR  = f"{WORKSPACE}/lerobot_dataset"

assert HF_TOKEN.startswith("hf_"), "set HF_TOKEN (env var or paste above) — PaliGemma is gated"

In [ ]:
# ── 2. Clone repo + install deps (idempotent, ~3-5 min first run) ─────────────
import subprocess, sys, pathlib

if not pathlib.Path(REPO_DIR, ".git").exists():
    url = GIT_URL.replace("https://", f"https://{GIT_TOKEN}@") if GIT_TOKEN else GIT_URL
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", f"{REPO_DIR}/requirements.txt"], check=True)
print("repo + deps ready")

In [ ]:
# ── 3. HuggingFace auth + gated-weights check ─────────────────────────────────
from huggingface_hub import login, whoami, auth_check
from huggingface_hub.errors import GatedRepoError

login(token=HF_TOKEN, add_to_git_credential=False)
print("logged in as:", whoami()["name"])

try:
    auth_check("google/paligemma-3b-pt-224")
    print("PaliGemma licence OK — gated weights accessible")
except GatedRepoError:
    raise SystemExit("PaliGemma is gated for this token — accept the licence at "
                     "https://huggingface.co/google/paligemma-3b-pt-224 and re-run")

In [ ]:
# ── 4. GPU check → auto memory mode + batch size ──────────────────────────────
import torch

assert torch.cuda.is_available(), "no CUDA GPU — pick a GPU pod"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
assert torch.cuda.is_bf16_supported(), "bf16 unsupported — use an Ampere+ GPU"
print(f"{name}  {vram:.0f} GB")

if MEMORY_MODE == "auto":
    MEMORY_MODE = "expert_only" if vram < 40 else ("freeze_vision" if vram < 70 else "full")
if BATCH_SIZE is None:
    BATCH_SIZE = {"expert_only": 4 if vram >= 40 else 2,
                  "freeze_vision": 4, "full": 4 if vram >= 70 else 2}[MEMORY_MODE]
print(f"memory_mode={MEMORY_MODE}  batch_size={BATCH_SIZE}")

In [ ]:
# ── 5. Pull the dataset from the Hub ──────────────────────────────────────────
from huggingface_hub import snapshot_download
import json, pathlib

snapshot_download(HF_DATASET_REPO, repo_type="dataset", local_dir=DATA_DIR)
info = json.loads(pathlib.Path(DATA_DIR, "meta", "info.json").read_text())
print(f"dataset OK — episodes={info['total_episodes']}  frames={info['total_frames']}  "
      f"fps={info['fps']}  robot={info.get('robot_type')}")

In [ ]:
# ── 6. Write the pod-local training config ────────────────────────────────────
import yaml, pathlib

base = yaml.safe_load(pathlib.Path(REPO_DIR, "policies", POLICY, "config.yaml").read_text())

base["dataset"]["root"] = DATA_DIR
base["model"].update({
    "dtype": "bfloat16",
    "gradient_checkpointing": True,
    "freeze_vision_encoder": MEMORY_MODE == "freeze_vision",
    "train_expert_only":     MEMORY_MODE == "expert_only",
    "proprio_mode": PROPRIO_MODE,
})
base["training"].update({
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "checkpoint_dir": f"policies/{POLICY}/checkpoints_runpod",
    "device": "cuda",
})

cfg_path = pathlib.Path(REPO_DIR, "policies", POLICY, "config.runpod.yaml")
cfg_path.write_text(yaml.safe_dump(base, sort_keys=False))
print("wrote", cfg_path)
print(yaml.safe_dump({"model-memory": {k: base["model"][k] for k in
      ("dtype", "gradient_checkpointing", "freeze_vision_encoder", "train_expert_only")},
      "training": base["training"]}, sort_keys=False))

In [ ]:
# ── 7. Launch training (survives notebook disconnects via nohup) ──────────────
# First run downloads PaliGemma (~5 GB) before step 1 — watch the log.
import subprocess, sys, pathlib

log = pathlib.Path(REPO_DIR, "train_pi05.log")
cmd = (f"cd {REPO_DIR} && nohup {sys.executable} common/train.py "
       f"--policy {POLICY} --config policies/{POLICY}/config.runpod.yaml "
       f"> {log} 2>&1 & echo $!")
pid = subprocess.check_output(cmd, shell=True, text=True).strip()
print(f"training started, pid={pid}\nlog: {log}\n"
      f"kill with: !kill {pid}")

In [ ]:
# ── 8. Monitor — re-run this cell any time ────────────────────────────────────
import pathlib, pandas as pd
import matplotlib.pyplot as plt

log = pathlib.Path(REPO_DIR, "train_pi05.log")
print("".join(log.read_text().splitlines(keepends=True)[-15:]))

csv = pathlib.Path(REPO_DIR, "policies", POLICY, "checkpoints_runpod", "metrics.csv")
if csv.exists():
    m = pd.read_csv(csv)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    m.plot(x="epoch", y=["train_l1", "val_l1"], ax=ax[0], title="loss (flow-matching MSE)")
    m.plot(x="epoch", y="grad_norm", ax=ax[1], title="grad norm")
    plt.tight_layout(); plt.show()
    best = m.loc[m.val_l1.idxmin()]
    print(f"best val_l1={best.val_l1:.4f} @ epoch {int(best.epoch)}")
else:
    print("metrics.csv not written yet (appears after epoch 1)")

In [ ]:
# ── 9. Ship checkpoints to the Hub (run when training is done) ────────────────
from huggingface_hub import HfApi, whoami
import pathlib

ckpt_dir = pathlib.Path(REPO_DIR, "policies", POLICY, "checkpoints_runpod")
repo = f"{whoami()['name']}/fr5-{POLICY}-{MEMORY_MODE}"

api = HfApi()
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=str(ckpt_dir), repo_id=repo,
                  allow_patterns=["best.pt", "metrics.csv"],
                  commit_message=f"{POLICY} {MEMORY_MODE} bs={BATCH_SIZE} epochs={MAX_EPOCHS}")
print(f"uploaded -> https://huggingface.co/{repo}")
print("deploy on the robot box: python common/deploy.py --checkpoint best.pt")

## Troubleshooting

**CUDA OOM** — in order of preference:
1. `MEMORY_MODE = "expert_only"` (freezes the 2B VLM; the 300M expert still learns the task)
2. halve `BATCH_SIZE` (set it explicitly, e.g. `BATCH_SIZE = 1`)
3. both

Both variables live in **cell 1 (Parameters)** — edit them there, kill the old run
(`!kill <pid>`), then re-run cells **1, 4, 6, 7** so the new values flow into the
auto-selection, the generated config, and the relaunch.

**`GatedRepoError` / 403 on PaliGemma** — the HF token's account hasn't accepted the licence, or the token lacks read scope.

**Pod restarted mid-run** — checkpoints land every `save_every` (10) epochs in
`policies/pi05/checkpoints_runpod/`; re-run cell 7 to continue from scratch weights
(the trainer has no resume flag yet) or lower `MAX_EPOCHS` to finish a shorter run.

**Throughput sanity** — with bf16 + gradient checkpointing on a 48 GB card expect roughly
1–3 s/step at batch 4 (π0.5, 150-episode dataset ⇒ a 100-epoch run is hours, not days).
Watch the first 50 steps for a *decreasing* `train_l1` before leaving it unattended.